In [1]:
import numpy as np
import pandas as pd

## Importing Dataset

In [2]:
df = pd.read_csv('C:/Users/Admin/DataScienceCourse/Amazon_Project/data/raw/amazon_products_sales_data_uncleaned.csv')

## 1. Summary Of Data
The dataset contains 42,675 Amazon product records with 16 attributes describing product details, pricing, customer engagement, availability, promotional indicators, URLs, and data collection information. It includes features such as product title, ratings, number of reviews, current and listed prices, best seller status, sponsorship, coupons, delivery information, sustainability badges, and product links. The dataset represents products across multiple categories and provides comprehensive information for analyzing product performance and marketplace trends.

Overall, the dataset contains a mixture of numerical, categorical, boolean, text, and URL-based features. While most essential product information is available, several columns contain missing values, particularly those related to sustainability badges, buy box availability, discounted price, delivery details, and recent purchase counts. The dataset captures diverse aspects of Amazon listings, making it suitable for exploring pricing patterns, customer feedback, promotional features, product availability, and overall sales-related characteristics.

## 2. Columns Description
#### **Table** -> `amazon_product_sales_data`:
1. `title` :- Contains the name or title of the product listed on Amazon. It serves as the primary identifier for each product.

2. `rating` :- Represents the average customer rating given to the product, typically on a scale of 1 to 5 stars.

3. `number_of_reviews` :- Shows the total number of customer reviews received by the product. This indicates the level of customer engagement and popularity.

4. `bought_in_last_month` :- Indicates the approximate number of units purchased during the last month. It provides an estimate of the product's recent sales activity.

5. `current/discounted_price` :- Represents the current selling price of the product after any applicable discounts or promotional offers.

6. `price_on_variant` :- Contains the price of a specific product variant, such as a different size, color, storage option, or model.

7. `listed_price` :- Shows the original or list price (MRP) of the product before discounts are applied.

8. `is_best_seller` :- Indicates whether the product has the Amazon Best Seller badge or not.

9. `is_sponsored` :- Indicates whether the product listing is promoted through Amazon Sponsored advertising.

10. `is_couponed` :- Specifies whether the product currently has a coupon or additional promotional discount available.

11. `buy_box_availability` :- Indicates whether the Amazon Buy Box is available for the product, allowing customers to purchase directly through the primary seller.

12. `delivery_details` :- Provides information about product delivery, such as estimated delivery dates, shipping speed, or delivery eligibility.

13. `sustainability_badges` :- Contains details about any sustainability or environmentally friendly certifications and badges assigned to the product.

14. `image_url` :- Stores the URL of the product's primary image displayed on the Amazon product page.

15. `product_url` :- Contains the direct URL linking to the product's page on Amazon.

16. `collected_at` :- Represents the date and time when the product information was collected or scraped from Amazon.

# Data Assessing

### Quality Issues

1. **model** - Encoding inconsistencies were observed in several text fields, where Unicode characters such as trademark symbols, quotation marks, and em dashes appeared as mojibake (e.g., â„¢, â€”). `validity` ✅  
2. **rating** - written in incorrect way like :'4.6 out of 5 stars' should be in float `validity`
3. **bought_in_last_month** - written in incorrect way should be in int instead of string `validity`
4. **Current/discounted price** - many value missing but can be replaced by price_on_variant col as basic price `validity`  
5. **price_on_variant** - should be in float instead of string `validity` 
6. **listed_price** - both  int and string values present, instead of 'No discount' can be replaced by price_variant & Current/discounted cols values `consistency`
7. **is_best_seller** - most values 90% values has no badge `accuracy`
8. **is_couponed** - most values has no coupon and Sometime give in Save in % and sometime Save in $ `accuracy`
9. **buy_box_availability** - null values `completeness`

10. **delivery_detail** -  Remove string other than dates ,time span given like : "Sep 1 - 11" , "Aug 29 - Sep 2" ,"FREE delivery Wed, Sep 3 on $35 of items shipped by AmazonOr fastest delivery Sun, Aug 31" `validity`
11. **product_url** - try for shortning url `consistency`

### Tidiness Issues
1. **listed_price** -  drop columns
2. **sustainability_badges** - try for dropping col , many missing values
3. **rename all columns name**

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   title                     42675 non-null  object
 1   rating                    41651 non-null  object
 2   number_of_reviews         41651 non-null  object
 3   bought_in_last_month      39458 non-null  object
 4   current/discounted_price  30926 non-null  object
 5   price_on_variant          42675 non-null  object
 6   listed_price              42675 non-null  object
 7   is_best_seller            42675 non-null  object
 8   is_sponsored              42675 non-null  object
 9   is_couponed               42675 non-null  object
 10  buy_box_availability      28022 non-null  object
 11  delivery_details          30955 non-null  object
 12  sustainability_badges     3408 non-null   object
 13  image_url                 42675 non-null  object
 14  product_url           

In [4]:
# Checking missing values
missing = df.isnull().mean()*100
print(missing[missing>0].sort_values(ascending=False))

sustainability_badges       92.014060
buy_box_availability        34.336262
current/discounted_price    27.531342
delivery_details            27.463386
bought_in_last_month         7.538371
product_url                  4.848272
rating                       2.399531
number_of_reviews            2.399531
dtype: float64


In [5]:
# checking Duplicated rows
n_dupes = df.duplicated().sum()
n_dupes
# no duplicated row

np.int64(0)

In [6]:
# Data overview
df.head()
df.tail()
df.sample()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
30352,"Duracell Coppertop 9V Battery, 6 Count (Pack o...",4.8 out of 5 stars,"5,257",20K+ bought in past month,19.69,basic variant price: nan,$23.26,Limited time deal,Sponsored,No Coupon,Add to cart,"FREE delivery Wed, Sep 3 on $35 of items shipp...",NaN,https://m.media-amazon.com/images/I/61ZSfHGJ2b...,NaN,2025-08-29 11:10:37


In [7]:
# Copying dataset
df1 = df.copy()

## Solving Validity Issue

In [8]:
# title col encoding issue 
df1.title.iloc[59]
# the "300™ -" shown as "300â„¢ -" while opening it in csv through excel
# by exporting to xlsx format it will be gone

'Sanus Height Adjustable Speaker Stand for Sonos Era 300™ - 17" Height Adjustment Sonos Stand Includes Carpet Spikes & Rubber Pads - Easy DIY Install Comes w/All Hardware - White'

In [9]:
# rating col
df1['rating'] = df1['rating'].str.split(' ').str.get(0).astype('float')

In [10]:
# number_of_reviews 
df1['number_of_reviews'] = df1['number_of_reviews'].str.replace(',','').fillna(0).astype('int')

In [11]:
# bought_in_last_month
df1['bought_in_last_month'] = df1['bought_in_last_month'].str.split('+').str.get(0).str.replace('K','000')
df1["bought_in_last_month"] = (df1["bought_in_last_month"].replace(r"^(?!\d+(\.\d+)?$).*$", np.nan, regex=True).infer_objects(copy=False))

In [12]:
# current/discounted price
df1['current/discounted_price'] = df1['current/discounted_price'].str.replace(',','').astype('float')

In [13]:
# price_on_variant
df1['price_on_variant'] = df1['price_on_variant'].str.split('$').str.get(1)
df1['price_on_variant'] = (df1['price_on_variant'].replace(r"^(?!\d+(\.\d+)?$).*$", np.nan, regex=True).infer_objects(copy=False).astype('float'))

In [14]:
# replaced current/discounted_price nan values with price_variant values
temp_df = df1[df1['current/discounted_price'].isna()]
df1.loc[temp_df.index,'current/discounted_price'] = temp_df['price_on_variant'].values

In [15]:
# replaced listed_price nan values with current/discounted_price values
temp_df = df1[df1['listed_price'] == 'No Discount']
df1.loc[temp_df.index,'listed_price'] = temp_df['current/discounted_price'].values

In [16]:
# removing $ and , sign from listed_price col 
df1["listed_price"] = (
    df1["listed_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
# Converting to numeric
df1["listed_price"] = pd.to_numeric(
    df1["listed_price"],
    errors="coerce"
)

In [17]:
# buy_box_availability
df1['buy_box_availability'] = df1['buy_box_availability'].fillna('Not Available')

In [18]:
# delivery_detail
import pandas as pd
import re

def extract_delivery_date(text):
    if pd.isna(text):
        return pd.NaT
    
    # Extract first month and day occurrence
    match = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(\d{1,2})', text)

    if match:
        month = match.group(1)
        day = match.group(2)

        date = pd.to_datetime(
            f"{month} {day} 2025",
            format="%b %d %Y"
        )

        return date
    
    return pd.NaT


df1['delivery_details'] = df1['delivery_details'].apply(extract_delivery_date)

In [19]:
# collected_at
df1['collected_at'] = pd.to_datetime(df1['collected_at'])

In [20]:
# dropping rows with multiple cols as nan value
cols = ["current/discounted_price","listed_price","delivery_details"] # dropped 2540 rows
temp_df = df1[df1[cols].isna().all(axis=1)]
temp_df
df1.drop(temp_df.index,inplace=True)

## Solving Tidiness issue

In [21]:
# dropping columns have most missing values
df1.drop(columns=['price_on_variant','sustainability_badges'],inplace=True)

In [22]:
# renaming all columns
df1.rename(columns={
    'title':'product_title','rating':'product_rating','number_of_reviews':'review_count',
    'bought_in_last_month':'units_sold_last_month','current/discounted_price':'discounted_price',
    'listed_price':'original_price','is_couponed':'has_couponed','delivery_details':'delivery_date',
    'collected_at':'data_collected_at'
},inplace=True)

In [23]:
# resetting index 
df1.reset_index(drop=True, inplace=True)

# Adding New Features

In [24]:
# Adding new product_category Column
category_keywords = {

    "Smartphones": [
        "iphone", "android phone", "galaxy s", "pixel", "smartphone"
    ],

    "Tablets": [
        "ipad", "tablet", "galaxy tab", "kindle"
    ],

    "Laptops": [
        "macbook", "laptop", "notebook", "chromebook"
    ],

    "Smartwatches": [
        "apple watch", "smartwatch", "galaxy watch"
    ],

    "Headphones & Earbuds": [
        "airpods", "earpods", "earbuds", "headphones",
        "headset", "buds", "beats", "sony", "jbl"
    ],

    "Microphones": [
        "microphone", "lavalier", "wireless mic",
        "condenser mic", "mic"
    ],

    "Batteries": [
        "battery", "batteries", "alkaline",
        "coin cell", "cr2032", "lithium battery",
        "aa", "aaa"
    ],

    "Storage Devices": [
        "hard drive", "hdd", "external hard drive",
        "portable drive", "nas", "flash drive",
        "usb drive"
    ],

    "Memory Cards": [
        "micro sd", "micro sdhc", "micro sdxc",
        "sd card", "memory card", "cfexpress"
    ],

    "Desktop Components": [
        "processor", "cpu", "ryzen", "intel core",
        "motherboard", "graphics card", "gpu",
        "ram", "nvme", "desktop"
    ],

    "Computer Accessories": [
        "keyboard", "mouse", "webcam",
        "monitor", "ssd", "usb"
    ],

    "Cables & Chargers": [
        "charger", "charging cable", "usb cable",
        "lightning cable", "usb-c", "usb c",
        "adapter", "power adapter"
    ],

    "Networking": [
        "router", "wifi", "wi-fi", "extender",
        "mesh", "modem", "switch", "tp-link"
    ],

    "Printers": [
        "printer", "officejet", "deskjet",
        "laserjet", "pixma", "ecotank"
    ],

    "Printer Ink": [
        "ink cartridge", "toner", "cyan",
        "magenta", "yellow", "black ink",
        "cartridge"
    ],

    "Calculators": [
        "calculator", "ti-84", "ti-30",
        "scientific calculator", "graphing calculator"
    ],

    "Camera Accessories": [
        "camera", "tripod", "camera strap",
        "gopro", "lens", "dslr"
    ],

    "Streaming Devices": [
        "roku", "fire tv", "chromecast",
        "streaming stick", "apple tv"
    ],

    "TV & Display Accessories": [
        "tv wall mount", "wall mount",
        "bracket", "display"
    ],

    "Office Supplies": [
        "paper", "laminating", "laminator",
        "folder", "folders", "marker",
        "markers", "pencil", "tape",
        "shipping tape", "packing tape"
    ],

    "Accessories": [
        "airtag", "apple pencil", "stylus",
        "case", "cover", "wallet",
        "mount"
    ],

    "Gaming Accessories": [
        "razer", "gaming", "rgb"
    ]
}

In [25]:
def extract_category(title):
    title = title.lower()

    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword in title:
                return category

    return "Other"


product_category = df1["product_title"].apply(extract_category)
df1.insert(1,'product_category',product_category)

In [26]:
# creating new month & day columns from delivery_date 
month = df1["delivery_date"].dt.month_name()
day = df1["delivery_date"].dt.day_name()

In [27]:
df1.insert(12,'delivery_month',month)
df1.insert(13,'delivery_day',day)

# Exporting Dataset

In [30]:
df1.to_csv('C:/Users/Admin/DataScienceCourse/Amazon_Project/data/Interim/amazon_product_sales_cleaned.csv',index=False)